# Algoritmo de Melkman 

In [9]:
def signo(p1, p2, p3):
    """
    Input: p1, p2, p3 (puntos (x, y))
    Output: número real (el doble del área con signo del triángulo p1 p2 p3)

    Indica hacia dónde se gira al recorrer p1->p2->p3:
        si > 0 izquierda (antihorario)
        si < 0 derecha (horario)
        si = 0 alineados
    """
    return (p2[0]-p1[0])*(p3[1]-p1[1]) - (p3[0]-p1[0])*(p2[1]-p1[1])

In [10]:
def MIN(P):
    """
    Input: P (lista de vértices)
    Output: el primer vértice de P

    Devuelve el primer vértice de P y lo elimina de la lista
    """
    return P.pop(0)

In [11]:
def ANTERIOR(s, C):
    """
    Input: s (vértice de C)
           C (lista de vértices del cierre)
    Output: vértice anterior a s en C

    C se trata como una lista circular: el anterior del primero es el último
    """
    return C[(C.index(s) - 1) % len(C)]

In [12]:
def POSTERIOR(s, C):
    """
    Input: s (vértice de C)
           C (lista de vértices del cierre)
    Output: vértice posterior a s en C

    C se trata como una lista circular: el posterior del último es el primero
    """
    return C[(C.index(s) + 1) % len(C)]

In [13]:
def en_Rq(p, q, C, O):
    """
    Input: p (punto a comprobar)
           q (último vértice añadido a C)
           C (cierre actual)
           O (orientación del triángulo inicial)
    Output: True si p está en R_q, False si no

    R_q es la región a la izquierda (según O) de las rectas q->POSTERIOR(q) y ANTERIOR(q)->q. 
    Si p está en R_q, no puede estar en el cierre
    """
    return O*signo(q, POSTERIOR(q, C), p) > 0 and O*signo(ANTERIOR(q, C), q, p) > 0

In [14]:
def melkman(P):
    """
    Input: P (lista de vértices [(x, y), ...] de un polígono simple, en orden)
    Output: C (vértices del cierre convexo)
            pasos (lista con lo que ocurre en cada paso, para la applet:
                   {'p': vértice, 'caso': 'inicio'/1/2, 'C': cierre tras el paso, 'borrados': vértices eliminados})

    Empieza con el triángulo de los 3 primeros vértices y añade los demás en orden:
        caso 1: p está en R_q -> se descarta
        caso 2: p no está en R_q -> entra en el cierre y se borran los vértices
                que quedan dentro
    """
    P = list(P) # copia, para no vaciar el polígono original
    pasos = []

    # Inicio: triángulo con los tres primeros vértices
    C = []
    p1 = MIN(P)
    p2 = MIN(P)
    q = MIN(P)
    C.extend([p1, p2, q])
    O = signo(p1, p2, q)
    pasos.append({'p': q, 'caso': 'inicio', 'C': list(C), 'borrados': []})

    while P != []:
        p = MIN(P)

        # Caso 1: p está en R_q -> se descarta
        while en_Rq(p, q, C, O) and P != []:
            pasos.append({'p': p, 'caso': 1, 'C': list(C), 'borrados': []})
            p = MIN(P)

        # Caso 2: p no está en R_q -> entra en el cierre
        if not en_Rq(p, q, C, O):
            borrados = []

            # Tangente hacia atrás: buscar el vértice s donde se apoya
            s = q
            t = ANTERIOR(s, C)
            while O*signo(t, s, p) <= 0:
                s = t
                t = ANTERIOR(s, C)

            C.insert(C.index(s) + 1, p)

            # Tangente hacia delante: borrar los vértices que quedan dentro
            s = POSTERIOR(p, C)
            t = POSTERIOR(s, C)
            while O*signo(s, t, p) <= 0:
                C.remove(s)
                borrados.append(s)
                s = t
                t = POSTERIOR(s, C)

            q = p
            pasos.append({'p': p, 'caso': 2, 'C': list(C), 'borrados': borrados})

        # El último vértice de P también estaba en R_q
        else:
            pasos.append({'p': p, 'caso': 1, 'C': list(C), 'borrados': []})

    return C, pasos

# Polígonos de ejemplo

In [15]:
import math

def ciudad(alturas=(3, 5, 2, 7, 4, 9, 3, 6, 2, 5)):
    """
    Input: alturas (altura de cada edificio)
    Output: lista de vértices del polígono

    Perfil de una ciudad: edificios de 2 de ancho separados por calles de 1 (a altura 1). 
    Recorre el suelo y luego los tejados de derecha a izquierda
    """
    n = len(alturas)
    P = [(0, 0), (3*n - 1, 0)] # suelo
    for j in reversed(range(n)):
        x = 3*j
        P += [(x + 2, alturas[j]), (x, alturas[j])] # tejado del edificio j
        if j > 0:
            P += [(x, 1), (x - 1, 1)] # bajar a la calle
    return P


def caracol(vueltas=1.75, n=28, ancho=0.6):
    """
    Input: vueltas (número de vueltas)
           n (puntos por curva)
           ancho (separación entre las dos curvas)
    Output: lista de vértices del polígono

    Banda en espiral: sale hacia fuera por una curva y vuelve hacia dentro por otra paralela
    """
    ida, vuelta = [], []
    for k in range(n+1):
        t = 2*math.pi*vueltas*k/n # ángulo
        r = 1 + 0.9*t # el radio crece con el ángulo
        ida.append((round(r*math.cos(t), 2), round(r*math.sin(t), 2)))
        vuelta.append((round((r+ancho)*math.cos(t), 2), round((r+ancho)*math.sin(t), 2)))
    return ida + vuelta[::-1]


# Ejemplos para el menú de la applet, ordenados por dificultad
ejemplos = {
    "1. Convexo":  [(0,0), (4,0), (5,3), (2,5), (-1,3)],
    "2. Estrella": [(2,0), (0.6,0.4), (0.6,1.9), (-0.5,0.6), (-1.6,1.2),
                    (-0.7,0), (-1.6,-1.2), (-0.5,-0.6), (0.6,-1.9), (0.6,-0.4)],
    "3. Ciudad":   ciudad(),
    "4. Caracol":  caracol(),
}